# Readme

what this section is about.


# Imports and Load Data

In [1]:
## load packages 
import jieba
import jieba.posseg as pseg
import jieba.analyse
import pandas as pd
import re
import numpy as np
import os
import ast
import pprint


## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer

## lda
from gensim import corpora
import gensim

## visualizing LDA
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
pyLDAvis.enable_notebook()

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

/opt/anaconda3/lib/python3.13/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
df = pd.read_csv("../data/02_tokenized.csv")
df.head()

,case_name,case_type,court,date,case_href,full_text,litigant_type,keywords
0,刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书,民事再审,吉林省高级人民法院,2021-11-05,https://wenshu.court.gov.cn/website/wenshu/181...,吉林省高级人民法院\n民 事 裁 定 书\n（2021）吉民再270号\n再审申请人（一审原...,individual,"['吉体', '劳动厅', '航模', '补上去', '人事厅', '劳动局', '机构编制..."
1,刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书,民事再审,山东省高级人民法院,2021-07-05,https://wenshu.court.gov.cn/website/wenshu/181...,山东省高级人民法院\n民 事 裁 定 书\n（2021）鲁民申3670号\n再审申请人（一审...,individual,"['历下区', '海蝶', '男因', '柴家', '书鲁民', '万床', '申号', '..."
2,马晓三、云南省人民检察院劳动争议再审民事判决书,民事再审,云南省高级人民法院,2020-06-09,https://wenshu.court.gov.cn/website/wenshu/181...,云南省高级人民法院\n民 事 判 决 书\n（2019）云民再30号\n抗诉机关：云南省人民...,individual,"['省体', '法制报', '云南省人民政府', '身份', '奖惩条例', '国家经贸委'..."
3,李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书,民事审判监督,北京市高级人民法院,2020-06-05,https://wenshu.court.gov.cn/website/wenshu/181...,北京市高级人民法院\n民 事 裁 定 书\n（2020）京民申2236号\n再审申请人（一审...,individual,"['人事科', '周润', '体育训练', '表中', '套改', '木樨园', '京民',..."
4,马俊岭等十三人故意伤害二审刑事附带民事判决书,刑事二审,陕西省高级人民法院,2018-07-30,https://wenshu.court.gov.cn/website/wenshu/181...,陕西省高级人民法院\n刑 事 附 带 民 事 判 决 书\n（2018）陕刑终147号\n原...,individual,"['洗杯', '枕部', '监控室', '侧脑室', '碑林区', '挫裂伤', '酒吧女'..."


In [3]:
# process into one giant list in preparation for DTM.
corpus = df.keywords.to_list()
corpus[:2] # ramdom check format

["['吉体', '劳动厅', '航模', '补上去', '人事厅', '劳动局', '机构编制', '慷本', '体工', '中有书吉民', '原省', '龙运', '现住', '退役费', '人王', '销应', '绿园区', '体工队', '工作失误', '工资表', '转递', '扩大会', '民事判决', '在编', '离队', '体委', '民事裁定', '人事处', '合法利益', '调令', '补报', '体育局', '人员名单', '体育事业', '离校', '处理意见', '团体冠军', '提审', '红头文件', '公章', '手写', '工作队', '委以', '证据不足', '单方', '名册', '文龙', '开庭审理', '转业', '偏袒']",
 "['历下区', '海蝶', '男因', '柴家', '书鲁民', '万床', '申号', '工商登记', '市中区', '民事判决', '文化课', '民事裁定', '通济', '调取', '体育局', '带病', '供货', '公司法', '法律效力', '录音', '涉案', '公示', '受害者', '退役', '占用', '挫折', '商贸', '纺织品', '民事', '注册资本', '可知', '增资', '办事处', '验证', '缴纳', '法定', '运动员', '资本金', '提交', '出资', '赔偿', '追加', '实质性', '推翻', '当事人', '目录', '无关', '注册', '认可', '足以']"]

# 2. Create a document-term matrix and do some basic diagnostics (more manual approach)

Here we'll create a DTM first using the raw documents; in the activity, you'll create one using the preprocessed docs
that you created in activity 1

## 2.1 Define the dtm function and select data to transform into a document-term matrix

In [4]:
## create DTM function
def create_dtm(list_of_strings, metadata):
    """ 
    Function to create dense document-term matrix (DTM) from a list of strings and provided metadata. 
    A sparse DTM is a list of term_index/doc_index tuples: if a given term occurs in a given doc at least once, 
        then this count is listed as a tuple; if not, that term/doc pair is omitted. 
    In a dense DTM, each row is one text (e.g., an Airbnb listing), each column is a term, and 
        each cell indicates the frequency of that word in that text. 
    
    Parameters:
        list_of_strings (Series): each row contains a preprocessed string (need not be tokenized)
        metadata (DataFrame): contains document-level covariates
    
    Returns:
        Dense DTM with metadata on left and then one column per word in lexicon
    """
    
    # initialize a sklearn tokenizer; this helps us tokenize the preprocessed string input
    vectorizer = CountVectorizer(lowercase = True) 
    dtm_sparse = vectorizer.fit_transform(list_of_strings)
    print('Sparse matrix form:\n', dtm_sparse[:3]) # take a look at sparse representation
    print()
    
    # switch the dataframe from the sparse representation to the normal dense representation (so we can treat it as regular dataframe)
    dtm_dense_named = pd.DataFrame(dtm_sparse.todense(), columns=vectorizer.get_feature_names_out ())
    print('Dense matrix form:\n', dtm_dense_named.head()) # take a look at dense representation
    dtm_dense_named_withid = pd.concat([metadata.reset_index(), dtm_dense_named], axis = 1) # add back document-level covariates

    return(dtm_dense_named_withid)

## 2.2 Execute the dtm function to create the document-term matrix

In [5]:
## example application on raw lowercase texts; 
dtm = create_dtm(list_of_strings = corpus,
                      metadata = df[['case_name', 'case_type', 'court', 'date']])

Sparse matrix form:
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 150 stored elements and shape (3, 2009)>
  Coords	Values
  (0, 517)	1
  (0, 410)	1
  (0, 1604)	1
  (0, 1644)	1
  (0, 122)	1
  (0, 412)	1
  (0, 1119)	1
  (0, 847)	1
  (0, 201)	1
  (0, 62)	1
  (0, 465)	1
  (0, 2008)	1
  (0, 1290)	1
  (0, 1894)	1
  (0, 144)	1
  (0, 1948)	1
  (0, 1515)	1
  (0, 202)	1
  (0, 730)	1
  (0, 748)	1
  (0, 1846)	1
  (0, 882)	1
  (0, 1190)	1
  (0, 574)	1
  (0, 1396)	1
  :	:
  (2, 643)	1
  (2, 898)	1
  (2, 1472)	1
  (2, 1068)	1
  (2, 1474)	1
  (2, 29)	1
  (2, 548)	1
  (2, 409)	1
  (2, 1347)	1
  (2, 464)	1
  (2, 1074)	1
  (2, 324)	1
  (2, 1557)	1
  (2, 508)	1
  (2, 1042)	1
  (2, 562)	1
  (2, 1870)	1
  (2, 415)	1
  (2, 460)	1
  (2, 732)	1
  (2, 669)	1
  (2, 1361)	1
  (2, 52)	1
  (2, 104)	1
  (2, 90)	1

Dense matrix form:
    一分钱  一贵  万床  丈八  三性  上位法  上受  上学  上报  上海市第一中级人民法院  ...  验证  骨关节  骨质增生  高分  \
0    0   0   0   0   0    0   0   0   0            0  ...   0    0     0   0   
1    0   

In [6]:
## show first set of rows/cols
dtm.head()

# slice out metadata
dtm.iloc[0:4, 5:]

,index,case_name,case_type,court,date,一分钱,一贵,万床,丈八,三性,...,验证,骨关节,骨质增生,高分,高欢,鲲鹏,麟游,黄标,默示,龙运
0,0,刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书,民事再审,吉林省高级人民法院,2021-11-05,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,1,刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书,民事再审,山东省高级人民法院,2021-07-05,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,2,马晓三、云南省人民检察院劳动争议再审民事判决书,民事再审,云南省高级人民法院,2020-06-09,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,3,李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书,民事审判监督,北京市高级人民法院,2020-06-05,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,马俊岭等十三人故意伤害二审刑事附带民事判决书,刑事二审,陕西省高级人民法院,2018-07-30,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


,一分钱,一贵,万床,丈八,三性,上位法,上受,上学,上报,上海市第一中级人民法院,...,验证,骨关节,骨质增生,高分,高欢,鲲鹏,麟游,黄标,默示,龙运
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,1,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
# basic summary stats of top words

## summing each col
top_terms = dtm[ dtm.columns[5:] ].sum(axis = 0)

## sorting from most frequent to least frequent
result = top_terms.sort_values(ascending = False)

pd.DataFrame(result).head(20)

,0
体育局,32
择业,23
民事判决,19
民事裁定,16
补偿费,15
补发,15
聘用,15
退役,14
体工队,14
运动队,14
